# BERT Masked Language Modeling with Hugging Face Transformers

이 노트북은 Kaggle Note 환경에서 `bert-base-uncased` 모델을 사용해 Masked Language Modeling을 수행하는 예제입니다.

구성:
- `AutoTokenizer`와 `AutoModelForMaskedLM` 사용
- `[MASK]` 토큰이 포함된 문장 입력
- PyTorch 기반 추론 수행
- `[MASK]` 위치에 대한 Top-5 예측 단어 출력

In [ ]:
# Kaggle 환경에서 transformers가 없을 수 있으므로 설치합니다.
# 이미 설치되어 있다면 빠르게 지나갑니다.
!pip -q install transformers

In [ ]:
# 필요한 라이브러리를 불러옵니다.
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

# 실행 장치를 설정합니다. Kaggle GPU가 켜져 있으면 CUDA를 사용합니다.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
# 사용할 사전학습 모델 이름을 지정합니다.
model_name = "bert-base-uncased"

# 토크나이저와 Masked Language Modeling 모델을 불러옵니다.
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name).to(device)
model.eval()

In [ ]:
# [MASK] 토큰이 포함된 입력 문장을 준비합니다.
# BERT는 반드시 tokenizer.mask_token, 즉 [MASK] 토큰을 사용해야 합니다.
text = "Paris is the [MASK] of France."
print("입력 문장:", text)
print("Mask token:", tokenizer.mask_token)

In [ ]:
# 입력 문장을 토큰화하고 PyTorch 텐서로 변환합니다.
inputs = tokenizer(text, return_tensors="pt")
inputs = {key: value.to(device) for key, value in inputs.items()}

# [MASK] 토큰의 위치를 찾습니다.
mask_token_index = (inputs["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]
print("[MASK] 위치:", mask_token_index.item())

In [ ]:
# PyTorch 기반으로 추론을 수행합니다.
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

# [MASK] 위치의 로짓만 추출한 뒤 Top-5 후보를 구합니다.
mask_token_logits = logits[0, mask_token_index, :]
top_k = 5
top_token_ids = torch.topk(mask_token_logits, top_k, dim=-1).indices[0].tolist()

print("[MASK] 위치 Top-5 예측 단어")
print("-" * 50)
for rank, token_id in enumerate(top_token_ids, start=1):
    token = tokenizer.decode([token_id]).strip()
    filled_text = text.replace(tokenizer.mask_token, token)
    print(f"{rank}. {token}")
    print(f"   예시 문장: {filled_text}")